<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex12.1-power-grid-stability-estimation/Ex12.1_pinn_grid_state.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the library files from the public course repository, so nothing needs uploading.*

<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Raissi, Perdikaris & Karniadakis, *Physics-informed neural networks*, J. Comput. Phys. 378 (2019) 686–707.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_12.1 — The State of a Grid

**Deep Learning for Engineering · Aalborg University · Part 2 · paired with L12.1 · Power Grid Stability Estimation**

A control room does not see the grid. It sees a few meters, and from them it
has to say what the voltage is at every bus. You do that two ways - the
estimator every utility runs, weighted least squares, and a network that
proposes the state and is judged by the power flow equations - and compare
them on accuracy and on time. Then the grid moves: a line faults, a machine
swings, and you find the machine's inertia from the frequency it recorded.

| | what you do | TODO |
|---|---|---|
| **1 – 3** | the problem, its data and its physics | — |
| **4** | the reference, and weighted least squares on three sets of meters for an agreed 0.01 p.u. | — |
| **5** | a network judged by the power flow equations, and the comparison | the power flow, the loss |
| **6** | the inverse problem: the inertia of a machine from a disturbance | — |
| **7** | what the notebook says | — |
| **8 – 9** | your report, and the mini project proposal | your answers |

About a minute and a half on a CPU. Numbers are printed with at most two
decimals, except errors smaller than that.

---

## 0 · Setup

In [ ]:
# files-cell v2 ----------------------------------------------------------
# The library files must sit beside the notebook. Locally they already do. On
# Google Colab they are fetched from the public course repository on every
# run, so a file left over from an earlier session cannot shadow it.
import os, sys, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex12.1-power-grid-stability-estimation/"
ON_COLAB = "google.colab" in sys.modules
for f in FILES:
    if ON_COLAB or not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
for _m in ("course_core", "pinn_core", "problem"):
    sys.modules.pop(_m, None)
print("files ready:", ", ".join(FILES))

In [ ]:
# --- setup ---------------------------------------------------------------
import os, time
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"
from pinn_core import *                                  # noqa: F401,F403
import course_core as cc
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt
import scipy.sparse as sp, scipy.sparse.linalg as spla

cc.OUTPUT_DIR = "Ex12.1_outputs"                              # where the report is written
set_seed(88)  # fix the random start so a run repeats
print("torch", torch.__version__, " device:", DEVICE, " dtype:", torch.get_default_dtype())

def middle(fn, reps=5):                                    # the middle of five timings, after a warm-up
    fn(); ts = []
    for _ in range(reps):
        t0 = time.perf_counter(); fn(); ts.append(time.perf_counter() - t0)
    return float(np.median(ts))

---

## 1 · The problem

Six buses joined by six lines: a link to the Nordic system (bus 0), a local
plant (bus 1), three load centres (buses 2 to 4) and an HVDC link (bus 5).
The **state** of the grid is the voltage at every bus: six magnitudes $V_i$
and five angles $\theta_i$ (the sixth angle is the reference, zero). Wanted:
those eleven numbers, from a handful of noisy meters.

## 2 · The data

| | |
|---|---|
| grid | 6 buses, 6 lines, per unit on a 100 MVA base |
| operating points | every injection between 0.6 and 1.2 of its nominal value |
| a PMU | reads $V$ to 0.004 p.u. and $\theta$ to 0.002 rad at its bus |
| a SCADA meter | reads the injections $P$ and $Q$ at its bus to 0.010 p.u. |
| a forecast | the injections at a bus with no meter, to 0.05 p.u. (assumed) |

Three sets of readings are compared:

| set | what is read | readings |
|---|---|---|
| `full` | PMUs at buses 0 and 3, injections at buses 0 to 4 | 14 |
| `thin` | a PMU at bus 0, injections at buses 0 and 3 | 6 |
| `forecast` | the thin set, and a forecast of the injections at buses 1, 2, 4 and 5 | 14 |

**The grid is representative of eastern Denmark, not a model of it.** Its
line impedances are textbook values, not measured ones: no operator publishes
those. The accuracy of a forecast is assumed.

## 3 · The physics

**The power flow equations** (L12.1's *The Power Flow Equations*). At every
bus $i$ the power injected equals the power that leaves along the lines:

$$P_i = V_i \sum_k V_k \big(G_{ik}\cos\theta_{ik} + B_{ik}\sin\theta_{ik}\big), \qquad
  Q_i = V_i \sum_k V_k \big(G_{ik}\sin\theta_{ik} - B_{ik}\cos\theta_{ik}\big)$$

with $\theta_{ik} = \theta_i - \theta_k$, and $G$ and $B$ the real and
imaginary parts of the admittance matrix: what the lines are. They are
algebraic. Nothing moves in them.

**Every reading is a function of the state.** A PMU reads $V_i$ and
$\theta_i$ directly; an injection meter reads $P_i$ or $Q_i$, which the
equations above give from the state. Call that function $h$. The cell below
draws the grid with the three sets of meters, prints the grid, its state at nominal load, and how many of the eleven
unknowns each set of readings determines.

In [ ]:
pb.draw_meters(); plt.show()                                 # the grid and the three sets of meters
pb.describe_problem()
Y = pb.build_ybus()                                           # the admittance matrix of the six lines

**What you should see.** Voltages between 0.971 and 1.000 p.u. and angles
within 2.5°, the city (bus 3) lowest. The full set and the forecast set
determine all 11 unknowns. **The thin set determines 5 of them**: six readings
cannot fix eleven numbers, however good the meters (L12.1's *Fewer
Measurements Than Unknowns*).

---

## 4 · The reference, and weighted least squares

**The reference** is the power flow itself. For 300 operating points the
injections are known, Newton's method solves the equations to $10^{-10}$, and
the state it returns is the truth. The readings are then made from it, with
noise.

**The classical estimator** is weighted least squares: the state that
minimises

$$J(x) = \sum_j \Big(\frac{z_j - h_j(x)}{\sigma_j}\Big)^2$$

by Gauss–Newton steps, where $z_j$ is reading $j$ and $\sigma_j$ its accuracy
(L12.1's *Weighted Least Squares, and What It Costs*). `pb.wls_all` runs it on
every operating point.

**The agreed accuracy is 0.01 p.u.**: the worst bus within 1 % of nominal
voltage, on average over the operating points. The cell below runs the
estimator on the three sets of readings.

In [ ]:
V_true, th_true = pb.scenarios(300, seed=7)                   # 300 operating points: the reference states
AGREED = 0.01

def score(V, th):
    # the worst bus of each operating point, averaged: magnitude in p.u., angle in degrees
    return float(np.abs(V - V_true).max(axis=1).mean()), float(np.degrees(np.abs(th - th_true).max(axis=1).mean()))

wls = {}
for kind in ("full", "thin", "forecast"):
    ms_k = pb.meters(kind)
    Z_k = pb.readings(V_true, th_true, ms_k, seed=3)          # what the meters read, with noise
    t0 = time.perf_counter(); V_k, th_k = pb.wls_all(Z_k, ms_k); sec = (time.perf_counter() - t0) / len(Z_k)
    wls[kind] = score(V_k, th_k) + (sec,)
    print(f"  weighted least squares, {kind:9s}: worst bus {wls[kind][0]:.4f} p.u. and {wls[kind][1]:.2f} deg, {1000 * sec:.2f} ms an estimate"
          + ("   <- meets 0.01" if wls[kind][0] <= AGREED else ""))

**What you should see.** With the full set the worst bus is within 0.003 p.u.
and 0.07°. **With the thin set it is off by 0.025 p.u. and 1.8°**, which is
more than the whole spread of voltages on this grid: the estimator returns
numbers, and at the buses with no meter they are close to a guess. With the
forecasts added it is within 0.005 p.u. and 0.15° again. An estimate takes
about a millisecond.

---

## 5 · A network judged by the power flow equations

**The network proposes the state; the equations judge it** (L12.1's *The
Network Learns the State, Not the Physics*). The network takes the 14 readings
of the forecast set and returns the 11 numbers of the state. Three layers of
64 tanh neurons.

**Two things are built in** (L12.1's *Build In What You Know for Certain*).
The reference angle is not an output: it is zero. The magnitudes pass through
a logistic function, so they lie between 0.90 and 1.10 p.u. whatever the
weights are.

**The loss needs no true state.** From the state the network proposes, the
power flow equations give what every meter should read, $h$. The loss is the
same $J$ as in section 4, averaged over 3000 operating points:

$$\mathcal{L} = \frac{1}{N}\sum_{n}\ \frac{1}{14}\sum_j \Big(\frac{z_j^{(n)} - h_j\big(\mathcal{N}(z^{(n)})\big)}{\sigma_j}\Big)^2$$

Weighted least squares minimises $J$ again for every new set of readings. The
network is trained once to return the minimiser for any of them: it
**amortises** the solve (L12.1's *What a Graph Network Adds, and What It Does
Not*). The network here is a dense one, because the grid never changes; a
graph network earns its place when the topology does.

**TODO 1** is the power flow in torch; **TODO 2** the loss.

In [ ]:
ms = pb.meters("forecast")                                    # the 14 readings the network is given
G, B = to_tensor(Y.real), to_tensor(Y.imag)                   # the admittance matrix, real and imaginary parts
SIGMA = to_tensor(ms.sigma).reshape(-1)                       # the accuracy of each reading

def power(V, th):
    # the power flow equations for a batch of states: V and th of shape (N, 6) -> P and Q of shape (N, 6)
    # TODO 1: the two sums of section 3, in torch, for every state of the batch at once.
    #
    #   dth = th[:, :, None] - th[:, None, :]                     # the angle differences theta_i - theta_k
    #   VV = V[:, :, None] * V[:, None, :]                        # the products V_i V_k
    #   P = (VV * (G * torch.cos(dth) + B * torch.sin(dth))).sum(dim=2)   # active power injected at each bus
    #   Q = (VV * (G * torch.sin(dth) - B * torch.cos(dth))).sum(dim=2)   # reactive power injected at each bus
    #   return P, Q
    raise NotImplementedError("Write the power flow equations")

def h(V, th):
    # what each meter of the set should read in the state (V, th): shape (N, 14)
    P, Q = power(V, th)
    return torch.stack([{"V": V, "TH": th, "P": P, "Q": Q}[kind][:, bus] for kind, bus in ms.spec], dim=1)

In [ ]:
class Estimator(torch.nn.Module):                             # a network: layers in __init__, the computation in forward
    def __init__(self):
        super().__init__()
        self.net = MLP(n_in=len(ms), n_out=2 * pb.N_BUS - 1, n_hidden=64, n_layers=3)   # dense tanh network: 14 readings in, 11 numbers out
    def forward(self, z):
        out = self.net(z)                                     # the network's 11 raw outputs
        V = 0.90 + 0.20 * torch.sigmoid(out[:, :pb.N_BUS])    # six magnitudes, between 0.90 and 1.10 p.u. by construction
        th = torch.cat([torch.zeros_like(out[:, :1]), 0.3 * out[:, pb.N_BUS:]], dim=1)   # the reference angle is zero; five more
        return V, th

V_train, th_train = pb.scenarios(3000, seed=1)                # 3000 operating points to train on
Z_train = pb.readings(V_train, th_train, ms, seed=5)          # their readings. The states are not used again.
mean, spread = Z_train.mean(axis=0), Z_train.std(axis=0) + 1e-6
scaled = lambda Z: to_tensor((Z - mean) / spread).to(DEVICE)  # the readings as the network takes them: zero mean, unit spread

In [ ]:
model = Estimator().to(DEVICE)                                # the network, on the device
Z_in, Z_obs = scaled(Z_train), to_tensor(Z_train).to(DEVICE)  # the readings, scaled for the network and as read

def loss_fn():
    # TODO 2: the loss of section 5 - no true state in it.
    #
    #   V, th = model(Z_in)                                       # the state the network proposes for each set of readings
    #   return (((Z_obs - h(V, th)) / SIGMA) ** 2).mean()         # what the meters should read against what they read
    raise NotImplementedError("Write the loss")

In [ ]:
print(f"{parameter_count(model)} weights, {len(Z_train)} sets of readings, no true state")
t0 = time.perf_counter()
# Adam first, then L-BFGS: the Part 2 training recipe
train_two_stage(model, loss_fn, adam_steps=1500, lbfgs_steps=30, lr=3e-3, report_every=0)
train_time = time.perf_counter() - t0
print(f"trained in {train_time:.0f} s")

In [ ]:
Z_test = pb.readings(V_true, th_true, ms, seed=3)             # the 300 held-out operating points of section 4
def estimate(Z):
    with torch.no_grad():  # no gradient tracking: evaluation only
        V, th = model(scaled(Z))                              # the network's state for each set of readings
        J = (((to_tensor(Z).to(DEVICE) - h(V, th)) / SIGMA) ** 2).mean(dim=1)   # its own check: how well the state explains the readings
    return to_numpy(V), to_numpy(th), to_numpy(J)
V_net, th_net, J_net = estimate(Z_test)
net_score = score(V_net, th_net)
t_one = middle(lambda: estimate(Z_test[:1]))                  # one estimate at a time
t_batch = middle(lambda: estimate(Z_test)) / len(Z_test)      # 300 at once

print(f"{'forecast set, 300 operating points':38s}{'worst |V|':>12s}{'worst angle':>13s}{'one estimate':>14s}{'training':>10s}")
print(f"  {'weighted least squares':36s}{wls['forecast'][0]:8.4f} p.u.{wls['forecast'][1]:9.2f} deg{1000 * wls['forecast'][2]:11.2f} ms{'-':>10s}")
print(f"  {'network, judged by the power flow':36s}{net_score[0]:8.4f} p.u.{net_score[1]:9.2f} deg{1000 * t_one:11.2f} ms{train_time:8.0f} s")
print(f"  the network on all 300 at once: {1e6 * t_batch:.1f} microseconds an estimate")

# a line the estimators are not told about: line 1-2 is out
Y_out = pb.build_ybus(outage=1)
V_out, th_out = pb.scenarios(100, seed=9, Y=Y_out)            # 100 operating points of the changed grid
Z_out = pb.readings(V_out, th_out, ms, seed=9, Y=Y_out)
worst = lambda V: float(np.abs(V - V_out).max(axis=1).mean())
V_a, _ = pb.wls_all(Z_out, ms, Y_out); V_b, _ = pb.wls_all(Z_out, ms, Y); V_c, _, J_out = estimate(Z_out)
print(f"line 1-2 out: least squares told of it {worst(V_a):.4f} p.u., not told {worst(V_b):.4f} p.u., the network {worst(V_c):.4f} p.u.")
print(f"the network's own check J: {J_net.mean():.2f} on the grid it was trained on, {J_out.mean():.2f} with the line out")

fig, ax = plt.subplots(1, 2, figsize=(10, 3.6))
ax[0].plot([0.93, 1.01], [0.93, 1.01], "k--", lw=1); ax[0].plot(V_true.ravel(), V_net.ravel(), ".", ms=2)
ax[0].set_xlabel("true |V|  [p.u.]"); ax[0].set_ylabel("the network's |V|  [p.u.]"); ax[0].set_title("300 operating points, six buses", fontsize=9)
ax[1].hist(J_net, bins=30, alpha=0.7, label="the grid it was trained on"); ax[1].hist(J_out, bins=30, alpha=0.7, label="line 1-2 out")
ax[1].set_xlabel("the network's own check J"); ax[1].legend(frameon=False, fontsize=8); plt.tight_layout()
plt.savefig(cc.output_path("Ex12.1_report.png"), dpi=150); plt.show()

**What you should see.** The network within about 0.004 p.u. and 0.15° on the
worst bus: **the same accuracy as weighted least squares**, from the same
readings, without ever having been shown a true state. One estimate takes a
quarter of a millisecond, and 300 at once about two microseconds each,
against a millisecond for least squares. Training took about 15 s.

**With a line out and nobody told**, both are wrong by about 0.05 p.u.: the
model of the grid is wrong, and both rest on it. Least squares told of the
outage is right again; the network would have to be trained again. But the
network's own check moves, from about 0.3 to about 2.4: the state it proposes
no longer explains the readings, and that can be seen with no truth at hand
(L12.1's *How You Know the Estimate Is Any Good*).

---

## 6 · The inverse problem: the inertia of a machine

**Now the grid moves** (L12.1's *The Swing Equation*). Line 0–1 faults at
0.5 s and is switched out at 0.6 s. The local plant swings against the rest of
the grid, by up to 28°. Its rotor angle $\delta$ and speed $\omega$ obey

$$\frac{d\delta}{dt} = \omega - \omega_s, \qquad
  \frac{2H}{\omega_s}\frac{d\omega}{dt} = P_m - P_e(\delta) - D\,\frac{\omega - \omega_s}{\omega_s}$$

with $P_m$ the mechanical power, $P_e(\delta)$ the electrical power it
delivers into the grid, $D$ the damping, $\omega_s$ the synchronous speed and
$H$ the **inertia constant** in seconds: the stored kinetic energy over the
rating. $H$ is the number wanted. It is 4.0 s in the record, which is
synthetic: a PMU at the plant, 50 readings of frequency a second, noise of
0.002 Hz.

**The model is the plant against a fixed grid.** In the record the grid is
not quite fixed (its own inertia is 150 s, not infinite), so the model is a
little wrong, as models are.

**Two ways again.** The classical one integrates the equation from the start
of the window for a trial $H$ and adjusts it by least squares, `pb.shoot_inertia`.
The physics-informed one trains a network for $\delta(t)$ and $\omega(t)$
**and** the number $H = e^{\beta}$ together: the loss is the residual of the
two equations plus the misfit to the recorded frequency. Each is started from
an inertia twice too large and from one twice too small, on the two seconds
after the fault is cleared - and then on the 0.45 s before the fault, when
nothing is happening.

In [ ]:
rec = pb.disturbance()                                        # the record: t, f, and the constants of the one-machine model
WS = rec["ws"]                                                # the synchronous speed, 2 pi 50 rad/s

class Swing(torch.nn.Module):                                 # a network for the swing, and one more trainable number
    def __init__(self, h_start):
        super().__init__()
        self.net = MLP(n_in=1, n_out=2, n_hidden=48, n_layers=4)    # dense tanh network: time in, angle and speed out
        self.beta = torch.nn.Parameter(torch.tensor(float(np.log(h_start))))   # H = exp(beta), trained beside the weights
    def forward(self, tau):
        out = self.net(tau)
        return out[:, :1], 5.0 * out[:, 1:]                   # the angle in rad, and the speed above synchronous in rad/s

def identify(lo, hi, h_start):
    # H from the record between lo and hi seconds, by the physics-informed fit started at h_start
    set_seed(88)                                              # fix the random start so a run repeats
    keep = (rec["t"] >= lo - 1e-9) & (rec["t"] <= hi + 1e-9)
    tau = to_tensor(((rec["t"][keep] - lo) / (hi - lo)).reshape(-1, 1), requires_grad=True).to(DEVICE)   # scaled time, differentiable
    f_obs = to_tensor(rec["f"][keep].reshape(-1, 1)).to(DEVICE)   # the recorded frequency
    sw = Swing(h_start).to(DEVICE)                            # the network and beta, on the device
    def swing_loss():
        d, x = sw(tau)                                        # the network's angle and speed
        H = torch.exp(sw.beta)                                # the inertia being identified
        r1 = grad(d, tau) / (hi - lo) - x                     # the first equation: d(delta)/dt = omega - omega_s
        r2 = 2 * H / WS * grad(x, tau) / (hi - lo) - (rec["Pm"] - pb.plant_power(d, rec) - rec["D"] * x / WS)   # the second, with H on the left
        return r1.pow(2).mean() + r2.pow(2).mean() + 1000.0 * (50.0 + x / (2 * np.pi) - f_obs).pow(2).mean()   # the residuals, and the misfit
    # Adam first, then L-BFGS: the Part 2 training recipe
    train_two_stage(sw, swing_loss, adam_steps=1000, lbfgs_steps=300, lr=3e-3, report_every=0)
    with torch.no_grad():  # no gradient tracking: evaluation only
        rms = float((50.0 + sw(tau)[1] / (2 * np.pi) - f_obs).pow(2).mean().sqrt())
    return float(torch.exp(sw.beta)), rms

inertia = {}
t0 = time.perf_counter()
for name, lo, hi in (("after the fault", 0.6, 2.6), ("before the fault", 0.0, 0.45)):
    keep = (rec["t"] >= lo - 1e-9) & (rec["t"] <= hi + 1e-9)
    for h_start in (8.0, 2.0):
        inertia[name, h_start] = pb.shoot_inertia(rec["t"][keep], rec["f"][keep], h_start, rec) + identify(lo, hi, h_start)
        hs, rs, hp, rp = inertia[name, h_start]
        print(f"  {name:17s} from {h_start:.0f} s: least squares H = {hs:5.2f} s (misfit {rs:.3f} Hz), physics-informed H = {hp:5.2f} s (misfit {rp:.3f} Hz)")
print(f"  the record was made with H = {rec['H_true']:.2f} s; all eight fits took {time.perf_counter() - t0:.0f} s")

**What you should see.** After the fault, **the physics-informed fit returns
3.89 s from both starts**, 3 % under the 4.00 s the record was made with, and
fits the record to its noise, 0.002 Hz. The 3 % is the model's error: the
grid was taken as fixed. **The classical fit stays near where it was
started**, 7.41 s and 1.83 s, with a misfit of 0.45 Hz. A swing integrated
with the wrong inertia rings at the wrong pitch, drifts out of step with the
record, and small changes of $H$ no longer improve it; started close to the
answer it finds 3.90 s.

Before the fault nothing moves, and **the four fits return 50.00, 1.94, 8.35
and 2.10 s, each with a perfect misfit**: near their own start, or at the
bound the fit was given. The record does not contain the inertia there, and
nothing in the fit says so (L12.1's *Inertia and Damping, Recovered
From a Disturbance*).

---

## 7 · What the notebook says

* **Six readings do not fix eleven numbers.** Weighted least squares on the
  thin set returned a state, and it was wrong at the buses with no meter.
  What rescued it was not a better algorithm but more information: a forecast
  of the injections, entered as a poor meter.
* **The network matched least squares without a single true state.** Its
  loss was the power flow equations applied to its own output. That is what
  physics-informed means here.
* **What the network bought was time.** The same accuracy from the same
  readings, in a fraction of the time once trained - which matters when the
  estimate is wanted for a thousand possible states of the grid, and not when
  it is wanted for one.
* **Both rest on the model of the grid.** A line out that nobody reported made
  both wrong by ten times. The network's own check, how well its state
  explains the readings, showed it. With no truth at hand, that check is what
  there is.
* **The inverse problem needed one more trainable number, and the equation
  written with it on the left.** Written as $\dot\omega = \ldots/H$, a very
  large $H$ makes the residual small whatever the swing does.
* **A fit always returns a number.** From the quiet window it returned a number
  near the one it was started with, or the bound it was allowed. Start a fit from two places before believing it.
* **The grid is not Denmark's.** Its impedances are textbook values and the
  accuracy of a forecast is assumed.

---

## 8 · Your report

**What these cells do.** You answer the questions below; the next cell writes `Ex12.1_report.md` with the results of section 5 and 6 and your answers, and the cell after it turns the report into a PDF for Moodle.

Each question builds part of an answer to one of L12.1's questions for the oral examination; the arrow says which. Keep each answer to a few sentences, with your numbers. Write your name in `NAME`, and each answer between the triple quotes that follow its question; an answer needs at least eight words.

In [ ]:
# TODO: write your answers. Every string below must be replaced.

NAME = "Your Name"

ANSWERS = {
    "The thin set has six readings for eleven unknowns. What did weighted least squares return at the buses with no meter, and what was it that made the estimate usable again? (-> L12.1 Q1)": """
""",
    "The network was never shown a true state. What was its loss, and what would it have learned if it had been given true states instead? With a line out, how did you see that its estimate was not to be trusted? (-> L12.1 Q2)": """
""",
    "What does the swing equation add to the power flow equations, and what in it sets how fast the plant swings? (-> L12.1 Q3)": """
""",
    "The two inertia fits were started from 8 s and from 2 s. Say what each returned after the fault and before it, and how you would know from the fits alone which window to believe. (-> L12.1 Q4)": """
""",
    "To conclude: which of the numbers in your tables would you show an operator, which would you not, and why? (-> L12.1, Exercise slide)": """
""",
}

In [ ]:
MIN_WORDS = 8                                               # an answer is a few sentences, not a word
short = [(i, len(text.split())) for i, text in enumerate(ANSWERS.values(), 1) if len(text.split()) < MIN_WORDS]
if NAME.strip() in ("", "Your Name") or short:
    print("Not ready - the report is written once these are fixed:")
    if NAME.strip() in ("", "Your Name"):
        print("  NAME: write your name")
    for i, n in short:
        print(f"  answer {i}: {n} word(s); write at least {MIN_WORDS}")
else:
    rows = ["| forecast set, 300 operating points | worst bus, magnitude | worst bus, angle | one estimate | training |", "|---|---|---|---|---|",
            f"| weighted least squares | {wls['forecast'][0]:.4f} p.u. | {wls['forecast'][1]:.2f} deg | {1000 * wls['forecast'][2]:.2f} ms | - |",
            f"| network, judged by the power flow | {net_score[0]:.4f} p.u. | {net_score[1]:.2f} deg | {1000 * t_one:.2f} ms | {train_time:.0f} s |",
            "", f"Weighted least squares: full set {wls['full'][0]:.4f} p.u., thin set {wls['thin'][0]:.4f} p.u.",
            "", f"Line 1-2 out: least squares told {worst(V_a):.4f} p.u., not told {worst(V_b):.4f} p.u., network {worst(V_c):.4f} p.u.; "
            f"the network's own check {J_net.mean():.2f} -> {J_out.mean():.2f}",
            "", "| window | started at | least squares H | physics-informed H |", "|---|---|---|---|"]
    rows += [f"| {name} | {h0:.0f} s | {v[0]:.2f} s | {v[2]:.2f} s |" for (name, h0), v in inertia.items()]
    rows += ["", f"The record was made with H = {rec['H_true']:.2f} s. The grid's impedances are textbook values, not measured."]
    out = ["# Ex_12.1 — The State of a Grid", "", f"**{NAME}**", "", "Deep Learning for Engineering · Aalborg University · 2026", "",
           "## Results", "", *rows, ""]
    for qq, text in ANSWERS.items():
        out += [f"## {qq}", "", text.strip(), ""]
    path = cc.output_path("Ex12.1_report.md")
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(out))
    print("written", path)

**What you should see.** `Not ready` until every answer is yours; then the path of the report. Then run the cell below: it turns the report into a PDF, with the figure of section 5, and downloads it. **Upload the PDF.**

In [ ]:
# Report as PDF for Moodle -----------------------------------------------
# Runs after the report cell above: turns Ex12.1_report.md into Ex12.1_report.pdf, with any figure
# saved as Ex12.1_report*.png embedded above the answers, and downloads it. Upload
# the PDF to Moodle; the .md stays as the source.
import subprocess, sys, glob, os
pdf_path = os.path.join(cc.OUTPUT_DIR, "Ex12.1_report.pdf")
try:
    import markdown, weasyprint
except ImportError:                       # installed already on a second run
    subprocess.run([sys.executable, "-m", "pip", "-q", "install", "markdown", "weasyprint"])
    import markdown, weasyprint

md_path = os.path.join(cc.OUTPUT_DIR, "Ex12.1_report.md")
if not os.path.exists(md_path):
    print("No report yet: it is written once every answer above is filled in.\n"
          "Fill them in, run the cell that writes the report, then run this one again.")
else:
    md = open(md_path, encoding="utf-8").read()
    figs = sorted(glob.glob("Ex12.1_report*.png")
                  + glob.glob(os.path.join(cc.OUTPUT_DIR, "Ex12.1_report*.png")))
    if figs:
        imgs = "\n\n".join(f"![{os.path.basename(p)}]({p})" for p in figs)
        i = md.find("\n## ", md.find("## Results") + 1) if "## Results" in md else -1
        md = (md[:i] + "\n\n" + imgs + "\n" + md[i:]) if i > 0 else md + "\n\n" + imgs + "\n"

    html = markdown.markdown(md, extensions=["fenced_code", "tables"])
    css = """body{font-family:Helvetica,Arial,sans-serif;font-size:11pt;margin:2cm}
    h1{font-size:18pt} h2{font-size:13pt;margin-top:18pt}
    pre{background:#f3f4f6;padding:8px;font-size:9.5pt} img{max-width:100%}"""
    weasyprint.HTML(string=f"<html><head><meta charset='utf-8'><style>{css}</style></head>"
                           f"<body>{html}</body></html>", base_url=".").write_pdf(pdf_path)
    print("written", pdf_path, f"with {len(figs)} figure(s)" if figs else "")
    try:
        from google.colab import files
        files.download(pdf_path)
    except ImportError:
        pass

---

## 9 · Mini project proposal

Two problems grown out of this set, for you to take further on your own. Each
student chooses **one** mini project from the Part 2 sets and has **one month**
to solve it individually. The dates and the hand-in are given in L13.

The ground truth is **given to you**, built by the course with a classical
solver and shown to have converged; the worked example below each project is
its first case. You do not solve the physics a second way: the month goes into
the deep learning - making a network predict what the ground truth says, as
accurately and as fast as the requirements ask.

### MP12.1A · A bigger grid, many machines

**The problem.** The set estimated one machine's trajectory on a six-bus network. A real area
has many machines swinging against each other after a fault, and meters at
only a few buses. Here the grid is the standard WSCC 3-machine, 9-bus system,
and the task is every machine's angle and speed through a fault.

| | |
|---|---|
| grid | WSCC 9-bus, 100 MVA, 60 Hz, data as in Anderson & Fouad (checked against the book before release) |
| machines | classical, $H$ = 23.64, 6.40, 3.01 s; $x'_d$ = 0.0608, 0.1198, 0.1813 p.u. |
| event | a three-phase fault at bus 7, cleared after 0.10 s by opening line 5-7 |
| window | 3 s |

**Why a plain PINN is not enough.** The machines are coupled through the network, so each one's swing equation
involves the others, and an unmetered machine is seen only through them.

**What you build.** A PINN for all three machines' angles and speeds, with their coupled swing
equations as the residual and the meters as data. You choose where the meters
go and show which machines they make observable.

**The ground truth you get.** `MP12.1A_truth.npz`, from `tools/miniprojects/ex121_truth.py` on
`grid9.py`: the power flow, the machines' internal voltages, the network
reduced onto them, RK4 at 0.5 ms; angles and speeds every 2 ms. Its power flow
reproduces the book's bus voltages and machine voltages (1.0566∠2.27°,
1.0502∠19.73°, 1.0170∠13.17°).

**Worked example.** The critical clearing time is 0.183 s, so the 0.10 s fault is survived:
machine 2 swings to 83.04° against machine 1, machine 3 to 60.22°. Steps of 2,
1 and 0.5 ms agree to 2.08e-08°.

**Requirements.**

* Every machine's angle within **1°** and speed within **0.01 Hz**, over the 3 s.
* The largest angle of machine 2 against 1 within **1°**.
* The smallest set of meters with which you meet both.

**What you hand in.** The error for each machine against the ground truth, which machines your meters
could not see, and the smallest set of meters that sees them all.

### MP12.1B · The inertia of every machine

**The problem.** Notebook 04 recovered one machine's inertia. As converters replace synchronous
machines, the inertia left in a grid falls and changes from hour to hour, and
operators need to know it. Here every machine's inertia and damping are
hidden, and the data are frequency records after a few disturbances.

| | |
|---|---|
| grid | the WSCC 9-bus system of MP12.1A |
| hidden | $H$ and $D$ of all three machines |
| events | faults at buses 7, 9 and 5, each cleared after 0.08 s by opening a line at it |
| records | each machine's frequency every 10 ms for 5 s, noise 1 mHz |

**Why a plain PINN is not enough.** Inertia and damping both slow a machine's response, and a disturbance near one
machine says little about another. Which disturbances separate them is part of
the answer.

**What you build.** An inverse PINN: the trajectories of all three machines with every $H$ and $D$
trainable, fitted to the swing equations and the frequency records. You show
which parameters each event identifies.

**The ground truth you get.** `MP12.1B_readings.npz` to fit and `MP12.1B_truth.npz` with the hidden
values, from the same script.

**Worked example.** A 10 % change of an inertia moves the records by 84 to 437 mHz; of a damping,
by only 7 to 31 mHz. The fault at bus 5 sees machine 3 least - 84 mHz for its
inertia, 7 mHz for its damping. The dampings are the hard half of the problem.

**Requirements.**

* Every inertia within **5 %**.
* Every damping within **20 %**, or a statement of which the data cannot give.
* The grid's total inertia within **2 %**.

**What you hand in.** The recovered inertias and dampings with uncertainties, the total inertia
against the true one, and the event that tells you least.